In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
from delta.tables import DeltaTable

In [0]:
product_desc = spark.read.table("databricksintech.silver.product_description")
product_desc.limit(3).display()

Product_description_id,Product_description,Modified_date
3,Chromoly steel.,2007-06-01
4,Aluminum alloy cups; large diameter spindle.,2007-06-01
5,Aluminum alloy cups and a hollow axle.,2007-06-01


In [0]:
product_m_d = spark.read.table("databricksintech.silver.product_model_description")
product_m_d.limit(3).display()

Product_model_id,Product_desc_id,Culture
1,1199,en
1,1467,ar
1,1589,fr


In [0]:
product_joined = product_desc.join(product_m_d, product_desc.Product_description_id==product_m_d.Product_desc_id)
product_joined = product_joined.drop("Modified_date", "Product_desc_id")
product_joined.limit(5).display()

Product_description_id,Product_description,Product_model_id,Culture
3,Chromoly steel.,95,en
4,Aluminum alloy cups; large diameter spindle.,96,en
5,Aluminum alloy cups and a hollow axle.,97,en
8,"Suitable for any type of riding, on or off-road. Fits any budget. Smooth-shifting with a comfortable ride.",23,en
64,"This bike delivers a high-level of performance on a budget. It is responsive and maneuverable, and offers peace-of-mind when you decide to go off-road.",22,en


In [0]:
product_joined.columns

['Product_description_id',
 'Product_description',
 'Product_model_id',
 'Culture']

In [0]:
(
    product_joined.write.format("delta")
    .mode("append") 
    .option(
        "path",
        "abfss://gold@intechaccstorage.dfs.core.windows.net/dim_product_desc",
    )
    .saveAsTable("databricksintech.gold.dim_product_desc")
)

In [0]:
# Reference target Delta table
gold_table = DeltaTable.forName(spark, "databricksintech.gold.dim_product_desc")

# Execute the Merge (Upsert)
gold_table.alias("target").merge(
    source = product_joined.alias("source"),
    condition = (
        "target.Product_description_id = source.Product_description_id"
        
    )
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Done!")

Done!
